# Generalisation Improvement
Step 1 - Download a sample of the 140k dataset via Kaggle API

Config - loads project paths once. Works from any machine/clone location; only `config.py` needs to exist at the repo root.

In [2]:
from config import BASE_DIR, DATASETS_DIR, DATASETS_FACESWAP_DIR, SAVED_MODELS_DIR, OUTPUTS_DIR, REPORTS_DIR


In [4]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

dataset = "xhlulu/140k-real-and-fake-faces"

files = api.dataset_list_files(dataset).files
print(f"Total files found: {len(files)}")
print(files[0].name if files else "none")

Total files found: 20
real_vs_fake/real-vs-fake/test/fake/00276TOPP4.jpg


Step 2 - Fetch full file list

In [2]:
import zipfile

zip_path = str(DATASETS_DIR / "140k-real-and-fake-faces.zip")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    names = zip_ref.namelist()
    print(f"Total entries in zip: {len(names)}")
    print(names[:10])

FileNotFoundError: [Errno 2] No such file or directory: 'E:/Study/Deepfake Detection Model/Deepfake-Detection-Model/datasets/140k-real-and-fake-faces.zip'

Step 3 - Sample and extract real + fake image from zip

filters the full 140,003-entry list down to just the train/real and train/fake subsets, then randomly samples 1,200 from each — same reproducible-sampling pattern we used for the FF++ videos.

In [2]:
import random
import os

random.seed(42)

train_real_entries = [n for n in names if n.startswith("real_vs_fake/real-vs-fake/train/real/")]
train_fake_entries = [n for n in names if n.startswith("real_vs_fake/real-vs-fake/train/fake/")]

print(f"Train real available: {len(train_real_entries)}")
print(f"Train fake available: {len(train_fake_entries)}")

SAMPLE_SIZE = 1200
sampled_real = random.sample(train_real_entries, min(SAMPLE_SIZE, len(train_real_entries)))
sampled_fake = random.sample(train_fake_entries, min(SAMPLE_SIZE, len(train_fake_entries)))

print(f"Sampling {len(sampled_real)} real + {len(sampled_fake)} fake images")

Train real available: 50000
Train fake available: 50000
Sampling 1200 real + 1200 fake images


Step 4 - Extract just the sampled files

What this does: opens the zip once, reads each sampled entry directly from inside the archive (zip_ref.open(entry)) and writes it out to disk — this way we never extract the full 140,003-file, 3.75GB archive, just the ~2,400 files we actually want. Much faster and saves disk space.

Note: tqdm needs to already be imported in this notebook — if you get a NameError, just add from tqdm import tqdm at the top first.

In [1]:
from tqdm import tqdm

extract_real_140k = str(DATASETS_DIR / "extra_real_140k")
extract_fake_140k = str(DATASETS_DIR / "extra_fake_140k")
os.makedirs(extract_real_140k, exist_ok=True)
os.makedirs(extract_fake_140k, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for entry in tqdm(sampled_real, desc="Extracting real"):
        filename = os.path.basename(entry)
        with zip_ref.open(entry) as source, open(os.path.join(extract_real_140k, filename), 'wb') as target:
            target.write(source.read())
    
    for entry in tqdm(sampled_fake, desc="Extracting fake"):
        filename = os.path.basename(entry)
        with zip_ref.open(entry) as source, open(os.path.join(extract_fake_140k, filename), 'wb') as target:
            target.write(source.read())

print(f"Extracted: {len(os.listdir(extract_real_140k))} real, {len(os.listdir(extract_fake_140k))} fake")

NameError: name 'DATASETS_DIR' is not defined

Step 5 - Copy the new images into your existing train folders

What this does: copies the 1,200+1,200 new images directly into your existing datasets/train/real and datasets/train/fake folders (which had 3,500 each from FF++) — so your training set becomes a genuine mix of professionally-filmed FF++ frames and everyday-style real/AI-generated photos.

In [ ]:
import shutil

train_real_dir = str(DATASETS_DIR / "train" / "real")
train_fake_dir = str(DATASETS_DIR / "train" / "fake")

for f in tqdm(os.listdir(extract_real_140k), desc="Merging real"):
    shutil.copy(os.path.join(extract_real_140k, f), os.path.join(train_real_dir, f))

for f in tqdm(os.listdir(extract_fake_140k), desc="Merging fake"):
    shutil.copy(os.path.join(extract_fake_140k, f), os.path.join(train_fake_dir, f))

print(f"Train real total now: {len(os.listdir(train_real_dir))}")
print(f"Train fake total now: {len(os.listdir(train_fake_dir))}")

Merging fake: 100%|██████████| 1200/1200 [00:18<00:00, 63.26it/s]

Train real total now: 4700
Train fake total now: 4700


Step 6 - Load the existing model and fine-tune on the expanded dataset


Key differences from original training:

weights=None then load_state_dict(...) with your existing efficientnet_best.pth — this continues training from where your model already is, not from scratch


Lower learning rate (0.00005 vs the original 0.0001) — since the model is already well-trained on FF++, we want smaller, gentler weight updates so it adds generalization capability without forgetting what it already learned well (a concept called catastrophic forgetting — worth knowing the term)

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 224
BATCH_SIZE = 16

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_dir = str(DATASETS_DIR)
train_data = datasets.ImageFolder(f"{data_dir}/train", transform=transform)
val_data = datasets.ImageFolder(f"{data_dir}/validation", transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

print("New train size:", len(train_data))
print("Classes:", train_data.classes)

# Load the ARCHITECTURE, then load your PREVIOUSLY TRAINED weights (not ImageNet weights this time)
model = models.efficientnet_b0(weights=None)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model.load_state_dict(torch.load(str(SAVED_MODELS_DIR / "efficientnet_best.pth"), map_location=device))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.00005)  # lower LR than original training — we're refining, not starting fresh

New train size: 6398
Classes: ['fake', 'real']


Step 7 - Fine-tuning loop (fewer epochs — this is refinement, not initial training)

Note: saves to a new file (efficientnet_finetuned.pth), not overwriting your original — so you keep both versions and can compare/revert if needed.

In [6]:
def train_one_epoch():
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def validate():
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

EPOCHS = 8
best_val_acc = 0.0

for epoch in range(EPOCHS):
    start = time.time()
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Time: {elapsed:.1f}s")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), str(SAVED_MODELS_DIR / "efficientnet_finetuned.pth"))
        print(f"  -> New best fine-tuned model saved (val_acc={val_acc:.4f})")

print("Fine-tuning complete. Best val accuracy:", best_val_acc)

KeyboardInterrupt: 

In [7]:
test_data = datasets.ImageFolder(f"{data_dir}/test", transform=transform)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Fine-tuned model on original FF++ test set:")
print(classification_report(all_labels, all_preds, target_names=test_data.classes))

Fine-tuned model on original FF++ test set:
              precision    recall  f1-score   support

        fake       0.97      0.99      0.98       621
        real       0.99      0.97      0.98       750

    accuracy                           0.98      1371
   macro avg       0.98      0.98      0.98      1371
weighted avg       0.98      0.98      0.98      1371

